In [58]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [59]:
import os
import shutil

db_path = "/content/drive/MyDrive/Demand_Forecasting_Inventory_Optimization/Database/demand_forecast.db"

local_db = "/content/demand_forecast.db"

if os.path.exists(db_path):
    shutil.copy(db_path, local_db)
    print("Database copied to local storage.")
else:
    raise FileNotFoundError("Database not found in Drive.")

print("Working Database:")
print(local_db)

Database copied to local storage.
Working Database:
/content/demand_forecast.db


In [60]:
import sqlite3

conn = sqlite3.connect("/content/demand_forecast.db")

cursor = conn.cursor()

print("Database connected successfully.")

Database connected successfully.


In [61]:
tables = pd.read_sql_query("""
SELECT name
FROM sqlite_master
WHERE type='table'
ORDER BY name
""", conn)

print("=== AVAILABLE TABLES ===")
print(tables)


required_tables = [
    "fact_daily_sales",
    "dim_item",
    "sku_error_summary",
    "fact_forecast_accuracy"
]

existing_tables = tables["name"].tolist()

for table in required_tables:
    assert table in existing_tables, f"{table} missing from database"


print("\nRequired tables validated successfully.")

=== AVAILABLE TABLES ===
                       name
0                  dim_date
1                  dim_item
2                 dim_model
3                 dim_store
4          fact_daily_sales
5    fact_forecast_accuracy
6     fact_inventory_policy
7  fact_service_level_sweep
8         sku_error_summary
9             stg_sales_raw

Required tables validated successfully.


In [62]:
inventory_base = pd.read_sql_query("""
    SELECT
        s.store_key,
        s.item_key,
        SUM(s.sales_qty) AS trailing_12m_demand,
        AVG(s.sales_qty) AS avg_daily_demand,
        e.best_sigma,
        e.naive_sigma,
        e.best_model_name,
        i.unit_price
    FROM fact_daily_sales s
    JOIN sku_error_summary e
        ON s.store_key = e.store_key
        AND s.item_key = e.item_key
    JOIN dim_item i
        ON s.item_key = i.item_key
    JOIN dim_date d
        ON s.date_key = d.date_key
    WHERE d.full_date BETWEEN '2016-10-01' AND '2017-09-30'
    GROUP BY
        s.store_key,
        s.item_key,
        e.best_sigma,
        e.naive_sigma,
        e.best_model_name,
        i.unit_price
""", conn)

print(inventory_base.head())

print("\nRows loaded:", len(inventory_base))

print("\nNull check:")
print(inventory_base.isnull().sum())

print("\nDistinct unit_price values:", inventory_base['unit_price'].nunique())

assert len(inventory_base) == 500
assert inventory_base.isnull().sum().sum() == 0
assert inventory_base['unit_price'].nunique() > 1

print("\ninventory_base loaded with real per-item pricing, validated.")

   store_key  item_key  trailing_12m_demand  avg_daily_demand  best_sigma  \
0          1         1                 8093         22.172603    4.755867   
1          1         2                21571         59.098630    7.293532   
2          1         3                13558         37.145205    5.925161   
3          1         4                 8085         22.150685    4.331804   
4          1         5                 6812         18.663014    4.286730   

   naive_sigma best_model_name  unit_price  
0     6.357586         XGBoost     2297.68  
1    10.559571         XGBoost      235.35  
2     8.582328         XGBoost      767.06  
3     6.557648         XGBoost     1783.76  
4     6.195378         XGBoost     2500.00  

Rows loaded: 500

Null check:
store_key              0
item_key               0
trailing_12m_demand    0
avg_daily_demand       0
best_sigma             0
naive_sigma            0
best_model_name        0
unit_price             0
dtype: int64

Distinct unit_price va

In [63]:
import numpy as np
inventory_base["annual_demand"] = inventory_base["trailing_12m_demand"]
inventory_base["ordering_cost"] = 750
inventory_base["holding_rate"] = 0.20
inventory_base["holding_cost"] = (
    inventory_base["unit_price"]
    *
    inventory_base["holding_rate"]
)

inventory_base["lead_time_days"] = 7

inventory_base["service_level"] = 0.95

inventory_base["z_value"] = 1.645


print(
    inventory_base[
        [
            "item_key",
            "unit_price",
            "holding_cost",
            "ordering_cost",
            "holding_rate",
            "lead_time_days",
            "service_level",
            "z_value"
        ]
    ].head(10)
)


print("\nParameters added successfully.")

print("\nValidation:")
print("Unique ordering cost:", inventory_base["ordering_cost"].unique())
print("Unique holding rate:", inventory_base["holding_rate"].unique())

   item_key  unit_price  holding_cost  ordering_cost  holding_rate  \
0         1     2297.68       459.536            750           0.2   
1         2      235.35        47.070            750           0.2   
2         3      767.06       153.412            750           0.2   
3         4     1783.76       356.752            750           0.2   
4         5     2500.00       500.000            750           0.2   
5         6      278.63        55.726            750           0.2   
6         7      256.08        51.216            750           0.2   
7         8       85.49        17.098            750           0.2   
8         9      303.16        60.632            750           0.2   
9        10       93.02        18.604            750           0.2   

   lead_time_days  service_level  z_value  
0               7           0.95    1.645  
1               7           0.95    1.645  
2               7           0.95    1.645  
3               7           0.95    1.645  
4        

In [65]:

import numpy as np

inventory_base["eoq"] = np.sqrt(
    (2 * inventory_base["annual_demand"] * inventory_base["ordering_cost"])
    / inventory_base["holding_cost"]
)

print(
    inventory_base[
        [
            "item_key",
            "annual_demand",
            "holding_cost",
            "eoq"
        ]
    ].drop_duplicates("item_key").head(10)
)

print("\nEOQ summary statistics:")
print(inventory_base["eoq"].describe())


mean_eoq = inventory_base["eoq"].mean()

print(f"\nMean EOQ: {mean_eoq:.1f}")


assert inventory_base["eoq"].nunique() > 1

assert 700 < mean_eoq < 1150, \
f"Mean EOQ {mean_eoq:.1f} is outside expected benchmark range"


print("\nEOQ calculated and validated against benchmark range.")

   item_key  annual_demand  holding_cost          eoq
0         1           8093       459.536   162.532655
1         2          21571        47.070   829.103350
2         3          13558       153.412   364.094215
3         4           8085       356.752   184.375170
4         5           6812       500.000   142.954538
5         6          21598        55.726   762.471232
6         7          21363        51.216   790.995334
7         8          28357        17.098  1577.259365
8         9          18889        60.632   683.595454
9        10          27110        18.604  1478.451979

EOQ summary statistics:
count     500.000000
mean      925.183175
std       705.490972
min       124.971997
25%       321.410200
50%       705.214807
75%      1386.374686
max      2930.891503
Name: eoq, dtype: float64

Mean EOQ: 925.2

EOQ calculated and validated against benchmark range.


In [66]:
inventory_base["naive_safety_stock"] = (
    inventory_base["z_value"]
    *
    inventory_base["naive_sigma"]
    *
    np.sqrt(inventory_base["lead_time_days"])
)

inventory_base["xgb_safety_stock"] = (
    inventory_base["z_value"]
    *
    inventory_base["best_sigma"]
    *
    np.sqrt(inventory_base["lead_time_days"])
)


inventory_base["naive_reorder_point"] = (
    inventory_base["avg_daily_demand"]
    *
    inventory_base["lead_time_days"]
    +
    inventory_base["naive_safety_stock"]
)

inventory_base["xgb_reorder_point"] = (
    inventory_base["avg_daily_demand"]
    *
    inventory_base["lead_time_days"]
    +
    inventory_base["xgb_safety_stock"]
)


print(
    inventory_base[
        [
            "item_key",
            "naive_sigma",
            "best_sigma",
            "naive_safety_stock",
            "xgb_safety_stock",
            "naive_reorder_point",
            "xgb_reorder_point"
        ]
    ].head(10)
)


total_naive_ss = inventory_base["naive_safety_stock"].sum()
total_xgb_ss = inventory_base["xgb_safety_stock"].sum()

ss_reduction_pct = (
    1 - total_xgb_ss / total_naive_ss
) * 100


print(f"\nTotal Naive Safety Stock: {total_naive_ss:,.2f}")
print(f"Total XGBoost Safety Stock: {total_xgb_ss:,.2f}")
print(f"Safety Stock Reduction: {ss_reduction_pct:.1f}%")


assert (
    inventory_base["xgb_safety_stock"]
    <
    inventory_base["naive_safety_stock"]
).all()

assert abs(ss_reduction_pct - 33.3) < 0.5


print("\nSafety Stock and Reorder Point calculated and validated for both policies.")

   item_key  naive_sigma  best_sigma  naive_safety_stock  xgb_safety_stock  \
0         1     6.357586    4.755867           27.669872         20.698772   
1         2    10.559571    7.293532           45.958008         31.743352   
2         3     8.582328    5.925161           37.352531         25.787848   
3         4     6.557648    4.331804           28.540595         18.853140   
4         5     6.195378    4.286730           26.963900         18.656969   
5         6    11.176621    7.985686           48.643572         34.755790   
6         7    11.889803    8.042468           51.747525         35.002921   
7         8    15.741203    9.539815           68.509823         41.519765   
8         9    11.682084    7.795199           50.843479         33.926742   
9        10    15.405508    9.741838           67.048789         42.399019   

   naive_reorder_point  xgb_reorder_point  
0           182.878091         175.906991  
1           459.648419         445.433763  
2        

In [67]:
# Cell 9 — Average Inventory and Annual Holding Cost

# Average Inventory = EOQ/2 + Safety Stock

inventory_base["naive_avg_inventory"] = (
    inventory_base["eoq"] / 2
    +
    inventory_base["naive_safety_stock"]
)

inventory_base["xgb_avg_inventory"] = (
    inventory_base["eoq"] / 2
    +
    inventory_base["xgb_safety_stock"]
)


# Annual Holding Cost

inventory_base["naive_holding_cost"] = (
    inventory_base["naive_avg_inventory"]
    *
    inventory_base["holding_cost"]
)

inventory_base["xgb_holding_cost"] = (
    inventory_base["xgb_avg_inventory"]
    *
    inventory_base["holding_cost"]
)


# Total portfolio holding cost

total_hc_naive = inventory_base["naive_holding_cost"].sum()

total_hc_xgb = inventory_base["xgb_holding_cost"].sum()

hc_delta = total_hc_naive - total_hc_xgb

hc_pct_total = (
    hc_delta / total_hc_naive
) * 100


# Safety stock attributable saving

ss_hc_naive = (
    inventory_base["naive_safety_stock"]
    *
    inventory_base["holding_cost"]
).sum()

ss_hc_xgb = (
    inventory_base["xgb_safety_stock"]
    *
    inventory_base["holding_cost"]
).sum()

ss_hc_pct = (
    (ss_hc_naive - ss_hc_xgb)
    /
    ss_hc_naive
) * 100


print("=== TOTAL PORTFOLIO HOLDING COST ===")
print(f"Naive:    ₹{total_hc_naive:,.0f}")
print(f"XGBoost:  ₹{total_hc_xgb:,.0f}")
print(f"Annual Saving: ₹{hc_delta:,.0f}")
print(f"Reduction: {hc_pct_total:.1f}%")


print("\n=== SAFETY-STOCK-ATTRIBUTABLE HOLDING COST ===")
print(f"Reduction: {ss_hc_pct:.1f}%")


assert abs(hc_pct_total - 5.3) < 0.5
assert abs(ss_hc_pct - 31.6) < 0.5
assert abs(hc_delta - 728902) < 5000


print("\nHolding cost validated against both locked benchmarks.")

=== TOTAL PORTFOLIO HOLDING COST ===
Naive:    ₹13,738,209
XGBoost:  ₹13,009,262
Annual Saving: ₹728,946
Reduction: 5.3%

=== SAFETY-STOCK-ATTRIBUTABLE HOLDING COST ===
Reduction: 31.6%

Holding cost validated against both locked benchmarks.


In [68]:
from scipy.stats import norm

service_levels = [0.90, 0.91, 0.92, 0.93, 0.94,
                  0.95, 0.96, 0.97, 0.98, 0.99]

sweep_results = []

for sl in service_levels:
    z = norm.ppf(sl)

    temp = inventory_base[
        [
            "store_key",
            "item_key",
            "avg_daily_demand",
            "lead_time_days",
            "naive_sigma",
            "best_sigma",
            "eoq",
            "holding_cost"
        ]
    ].copy()

    temp["service_level"] = sl
    temp["z_value"] = z

    temp["naive_safety_stock"] = (
        z *
        temp["naive_sigma"] *
        np.sqrt(temp["lead_time_days"])
    )

    temp["xgb_safety_stock"] = (
        z *
        temp["best_sigma"] *
        np.sqrt(temp["lead_time_days"])
    )


    temp["naive_reorder_point"] = (
        temp["avg_daily_demand"] *
        temp["lead_time_days"]
        +
        temp["naive_safety_stock"]
    )

    temp["xgb_reorder_point"] = (
        temp["avg_daily_demand"] *
        temp["lead_time_days"]
        +
        temp["xgb_safety_stock"]
    )


    temp["naive_avg_inventory"] = (
        temp["eoq"] / 2
        +
        temp["naive_safety_stock"]
    )

    temp["xgb_avg_inventory"] = (
        temp["eoq"] / 2
        +
        temp["xgb_safety_stock"]
    )


    temp["naive_holding_cost"] = (
        temp["naive_avg_inventory"]
        *
        temp["holding_cost"]
    )

    temp["xgb_holding_cost"] = (
        temp["xgb_avg_inventory"]
        *
        temp["holding_cost"]
    )


    sweep_results.append(temp)


service_level_df = pd.concat(
    sweep_results,
    ignore_index=True
)


print(f"Rows generated: {len(service_level_df)}")


sweep_summary = service_level_df.groupby("service_level").agg(
    avg_naive_ss=("naive_safety_stock", "mean"),
    avg_xgb_ss=("xgb_safety_stock", "mean"),
    total_naive_hc=("naive_holding_cost", "sum"),
    total_xgb_hc=("xgb_holding_cost", "sum")
)

print("\n", sweep_summary.round(1))


is_naive_monotonic = (
    sweep_summary["avg_naive_ss"]
    .is_monotonic_increasing
)

is_xgb_monotonic = (
    sweep_summary["avg_xgb_ss"]
    .is_monotonic_increasing
)

xgb_below_naive_always = (
    service_level_df["xgb_safety_stock"]
    <
    service_level_df["naive_safety_stock"]
).all()


print("\nNaive monotonic:", is_naive_monotonic)
print("XGBoost monotonic:", is_xgb_monotonic)
print("XGBoost below naive:", xgb_below_naive_always)


sl95 = service_level_df[
    service_level_df.service_level == 0.95
]

print("\n95% slice cross-check:")

print(
    f"Naive SS: {sl95['naive_safety_stock'].sum():,.1f}"
)

print(
    f"XGB SS: {sl95['xgb_safety_stock'].sum():,.1f}"
)

print(
    f"Naive HC: ₹{sl95['naive_holding_cost'].sum():,.0f}"
)

print(
    f"XGB HC: ₹{sl95['xgb_holding_cost'].sum():,.0f}"
)


assert len(service_level_df) == 5000
assert is_naive_monotonic
assert is_xgb_monotonic
assert xgb_below_naive_always


print("\nService-level sweep validated.")

Rows generated: 5000

                avg_naive_ss  avg_xgb_ss  total_naive_hc  total_xgb_hc
service_level                                                        
0.90                   39.1        26.1      13228779.3    12660887.3
0.91                   40.9        27.3      13311762.1    12717635.4
0.92                   42.9        28.6      13401911.7    12779284.4
0.93                   45.1        30.0      13501036.0    12847070.9
0.94                   47.5        31.7      13611742.3    12922777.8
0.95                   50.2        33.5      13738003.4    13009121.8
0.96                   53.5        35.6      13886343.9    13110565.0
0.97                   57.4        38.3      14068709.8    13235276.5
0.98                   62.7        41.8      14311133.4    13401058.7
0.99                   71.0        47.4      14693223.0    13662351.9

Naive monotonic: True
XGBoost monotonic: True
XGBoost below naive: True

95% slice cross-check:
Naive SS: 25,114.8
XGB SS: 16,744.5
Naiv

In [69]:
cursor.execute("DROP TABLE IF EXISTS fact_inventory_policy")
cursor.execute("""
CREATE TABLE fact_inventory_policy (
    store_key INTEGER NOT NULL,
    item_key INTEGER NOT NULL,
    avg_daily_demand REAL,
    annual_demand REAL,
    unit_price REAL,
    eoq REAL,
    naive_sigma REAL,
    best_sigma REAL,
    best_model_name TEXT,
    naive_safety_stock REAL,
    xgb_safety_stock REAL,
    naive_reorder_point REAL,
    xgb_reorder_point REAL,
    naive_avg_inventory REAL,
    xgb_avg_inventory REAL,
    naive_holding_cost REAL,
    xgb_holding_cost REAL,
    holding_cost_saving REAL,
    service_level REAL,
    z_value REAL,
    PRIMARY KEY (store_key, item_key)
)
""")

conn.commit()


inventory_base["holding_cost_saving"] = (
    inventory_base["naive_holding_cost"]
    -
    inventory_base["xgb_holding_cost"]
)


policy_cols = [
    "store_key",
    "item_key",
    "avg_daily_demand",
    "annual_demand",
    "unit_price",
    "eoq",
    "naive_sigma",
    "best_sigma",
    "best_model_name",
    "naive_safety_stock",
    "xgb_safety_stock",
    "naive_reorder_point",
    "xgb_reorder_point",
    "naive_avg_inventory",
    "xgb_avg_inventory",
    "naive_holding_cost",
    "xgb_holding_cost",
    "holding_cost_saving",
    "service_level",
    "z_value"
]


cursor.executemany(
    f"""
    INSERT INTO fact_inventory_policy
    ({','.join(policy_cols)})
    VALUES ({','.join(['?'] * len(policy_cols))})
    """,
    inventory_base[policy_cols].itertuples(index=False, name=None)
)

conn.commit()

cursor.execute("DROP TABLE IF EXISTS fact_service_level_sweep")

cursor.execute("""
CREATE TABLE fact_service_level_sweep (
    store_key INTEGER NOT NULL,
    item_key INTEGER NOT NULL,
    service_level REAL NOT NULL,
    z_value REAL,
    naive_safety_stock REAL,
    xgb_safety_stock REAL,
    naive_reorder_point REAL,
    xgb_reorder_point REAL,
    naive_avg_inventory REAL,
    xgb_avg_inventory REAL,
    naive_holding_cost REAL,
    xgb_holding_cost REAL,
    PRIMARY KEY (store_key, item_key, service_level)
)
""")

conn.commit()


sweep_cols = [
    "store_key",
    "item_key",
    "service_level",
    "z_value",
    "naive_safety_stock",
    "xgb_safety_stock",
    "naive_reorder_point",
    "xgb_reorder_point",
    "naive_avg_inventory",
    "xgb_avg_inventory",
    "naive_holding_cost",
    "xgb_holding_cost"
]


cursor.executemany(
    f"""
    INSERT INTO fact_service_level_sweep
    ({','.join(sweep_cols)})
    VALUES ({','.join(['?'] * len(sweep_cols))})
    """,
    service_level_df[sweep_cols].itertuples(index=False, name=None)
)

conn.commit()
n_policy = pd.read_sql_query(
    "SELECT COUNT(*) as n FROM fact_inventory_policy",
    conn
)["n"][0]
n_sweep = pd.read_sql_query(
    "SELECT COUNT(*) as n FROM fact_service_level_sweep",
    conn
)["n"][0]


total_saving = pd.read_sql_query(
    "SELECT SUM(holding_cost_saving) as s FROM fact_inventory_policy",
    conn
)["s"][0]


print(f"fact_inventory_policy rows: {n_policy}")

print(f"fact_service_level_sweep rows: {n_sweep}")

print(
    f"Total holding cost saving from DB: ₹{total_saving:,.0f}"
)


assert n_policy == 500
assert n_sweep == 5000
assert abs(total_saving - 728946) < 5000


checkpoint_to_drive()

print("\nBoth tables created, validated, and checkpointed to Drive.")

fact_inventory_policy rows: 500
fact_service_level_sweep rows: 5000
Total holding cost saving from DB: ₹728,946
Database copied back to Google Drive.

Both tables created, validated, and checkpointed to Drive.
